# 📊 BookBuddy: Comparative LLM Evaluation Framework

## Objective
Compare BookBuddy's recommendation quality across multiple LLM backends:
- **Commercial**: GPT-4, GPT-3.5
- **Current**: Qwen2.5-7B-Instruct (baseline)
- **Open-Source**: Llama-2-7B, Mistral-7B, Phi-3-medium

## Evaluation Dimensions
1. **Response Quality**: Personalization, reasoning, clarity
2. **Reliability**: Hallucination rate, consistency
3. **Efficiency**: Inference time, token usage
4. **Cost**: Per-user cost analysis
5. **Pedagogical Fit**: Appropriateness for educational recommendations


## Setup & Dependencies

In [ ]:
import subprocess
import sys

packages = [
    'openai',
    'transformers',
    'torch',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'tqdm',
    'accelerate',
    'bitsandbytes',
    'scipy',
]

for package in packages:
    try:
        __import__(package)
        print(f'✓ {package} already installed')
    except ImportError:
        print(f'Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print('\n✅ All dependencies installed!')

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print('📦 Core libraries loaded successfully')

## Phase 1: Load BookBuddy Artifacts & Helper Functions

In [ ]:
ARTIFACTS_DIR = './bookbuddy_artifacts'
MODELS_DIR = './models'
EVALUATIONS_DIR = './evaluations'

print('📂 Loading BookBuddy artifacts...')

try:
    books_df = pd.read_parquet(f'{ARTIFACTS_DIR}/books_clean.parquet')
    ratings_df = pd.read_parquet(f'{ARTIFACTS_DIR}/ratings_filtered.parquet')
    test_df = pd.read_parquet(f'{ARTIFACTS_DIR}/test.parquet')
    
    with open(f'{ARTIFACTS_DIR}/artifacts.pkl', 'rb') as f:
        artifacts = pickle.load(f)
    
    user_map = artifacts['user_map']
    book_map = artifacts['book_map']
    id_col = artifacts['id_col']
    
    print(f'✓ Loaded {len(books_df)} books')
    print(f'✓ Loaded {len(ratings_df)} ratings')
    print(f'✓ Loaded mappings (users: {len(user_map)}, books: {len(book_map)})')
except FileNotFoundError as e:
    print(f'⚠ Warning: {e}')
    print('Make sure you run preprocessing.ipynb first')

In [ ]:
def create_test_users(ratings_df, sample_size=20):
    """
    Create diverse test set of users across different interaction counts.
    """
    user_stats = ratings_df.groupby('user_id').agg({
        'rating': ['count', 'mean'],
        'book_id': lambda x: x.nunique()
    }).reset_index()
    user_stats.columns = ['user_id', 'interaction_count', 'avg_rating', 'unique_books']
    
    user_stats['interaction_tier'] = pd.cut(
        user_stats['interaction_count'], 
        bins=[0, 10, 50, 200, float('inf')],
        labels=['Cold (0-10)', 'Warm (10-50)', 'Medium (50-200)', 'Hot (200+)']
    )
    
    test_users = []
    for tier in ['Cold (0-10)', 'Warm (10-50)', 'Medium (50-200)', 'Hot (200+)']:
        tier_users = user_stats[user_stats['interaction_tier'] == tier].sample(
            n=min(sample_size // 4, len(user_stats[user_stats['interaction_tier'] == tier])),
            random_state=42
        )
        test_users.extend(tier_users['user_id'].tolist())
    
    return test_users, user_stats

try:
    test_user_ids, user_stats = create_test_users(ratings_df, sample_size=20)
    print(f'\n✅ Created test set with {len(test_user_ids)} diverse users')
    print(user_stats[user_stats['user_id'].isin(test_user_ids)][['user_id', 'interaction_count', 'interaction_tier']].head(10))
except:
    print('⚠ Could not create test users. Load data first.')

In [ ]:
def get_user_recommendation_context(user_id, ratings_df, books_df, num_recent=5):
    """
    Get user reading history and proficiency profile.
    """
    user_ratings = ratings_df[ratings_df['user_id'] == user_id].sort_values('rating', ascending=False)
    recent_books = user_ratings.head(num_recent).merge(books_df, on='book_id')
    
    proficiency_dist = recent_books['proficiency_level'].value_counts()
    primary_proficiency = proficiency_dist.index[0] if len(proficiency_dist) > 0 else 'Intermediate (B1)'
    
    context = {
        'user_id': user_id,
        'total_books_read': len(user_ratings),
        'avg_rating': user_ratings['rating'].mean(),
        'primary_proficiency': primary_proficiency,
        'recent_books': recent_books[['title', 'authors', 'proficiency_level', 'rating']].to_dict('records'),
    }
    
    return context

def create_llm_prompt_template(user_context, top_candidates):
    """
    Create standardized prompt for all LLMs.
    """
    recent_books_str = json.dumps(user_context['recent_books'][:3], indent=2)
    candidates_str = json.dumps(top_candidates[:10], indent=2)
    
    prompt = f"""You are a knowledgeable book recommendation agent. Based on the user's reading history and proficiency level, 
provide personalized recommendations with clear pedagogical reasoning.

USER PROFILE:
- Total books read: {user_context['total_books_read']}
- Reading proficiency level: {user_context['primary_proficiency']}
- Average rating given: {user_context['avg_rating']:.2f}/5

RECENT READING HISTORY:
{recent_books_str}

TOP 10 CANDIDATE RECOMMENDATIONS:
{candidates_str}

Please select the TOP 3 most suitable recommendations and explain your reasoning based on:
1. Alignment with user's proficiency level
2. Match with demonstrated reading interests
3. Progression difficulty (pedagogically appropriate)

Format your response as a numbered list with explanations.
"""
    return prompt

print('✅ Helper functions defined')

## Phase 2: LLM Backend Implementations

In [ ]:
class LLMBackend:
    """
    Abstract base class for LLM backends.
    """
    def __init__(self, model_name):
        self.model_name = model_name
        self.load_time = None
        self.inference_times = []
        self.token_counts = []
    
    def load_model(self):
        raise NotImplementedError
    
    def generate_recommendation(self, prompt, max_tokens=500):
        raise NotImplementedError
    
    def get_metrics(self):
        return {
            'model_name': self.model_name,
            'avg_inference_time': np.mean(self.inference_times) if self.inference_times else 0,
            'avg_tokens': np.mean(self.token_counts) if self.token_counts else 0,
            'total_runs': len(self.inference_times),
        }

print('✅ LLMBackend abstract class defined')

In [ ]:
class QwenBackend(LLMBackend):
    """
    Qwen2.5-7B-Instruct backend (current BookBuddy implementation).
    """
    def __init__(self):
        super().__init__('Qwen2.5-7B-Instruct')
        self.model = None
        self.tokenizer = None
        self.device = 'cuda'
    
    def load_model(self):
        print(f'Loading {self.model_name}...')
        start_time = time.time()
        
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            import torch
            
            self.tokenizer = AutoTokenizer.from_pretrained(
                'Qwen/Qwen2.5-7B-Instruct',
                trust_remote_code=True
            )
            
            self.model = AutoModelForCausalLM.from_pretrained(
                'Qwen/Qwen2.5-7B-Instruct',
                trust_remote_code=True,
                load_in_4bit=True,
                device_map='auto'
            )
            
            self.load_time = time.time() - start_time
            print(f'✓ {self.model_name} loaded in {self.load_time:.2f}s')
            return True
        except Exception as e:
            print(f'✗ Failed to load {self.model_name}: {e}')
            return False
    
    def generate_recommendation(self, prompt, max_tokens=500):
        if self.model is None:
            return None, 0, 0
        
        start_time = time.time()
        try:
            import torch
            inputs = self.tokenizer.encode(prompt, return_tensors='pt').to(self.device)
            
            outputs = self.model.generate(
                inputs,
                max_new_tokens=max_tokens,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
            
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            inference_time = time.time() - start_time
            token_count = len(outputs[0])
            
            self.inference_times.append(inference_time)
            self.token_counts.append(token_count)
            
            return response, inference_time, token_count
        except Exception as e:
            print(f'Error generating recommendation: {e}')
            return None, 0, 0

print('✅ QwenBackend class defined')

In [ ]:
class OpenSourceLLMBackend(LLMBackend):
    """
    Generic backend for open-source LLMs from Hugging Face.
    """
    def __init__(self, model_name, hf_model_id):
        super().__init__(model_name)
        self.hf_model_id = hf_model_id
        self.model = None
        self.tokenizer = None
        self.device = 'cuda'
    
    def load_model(self):
        print(f'Loading {self.model_name} ({self.hf_model_id})...')
        start_time = time.time()
        
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            
            self.tokenizer = AutoTokenizer.from_pretrained(self.hf_model_id, trust_remote_code=True)
            
            self.model = AutoModelForCausalLM.from_pretrained(
                self.hf_model_id,
                trust_remote_code=True,
                load_in_4bit=True,
                device_map='auto'
            )
            
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            
            self.load_time = time.time() - start_time
            print(f'✓ {self.model_name} loaded in {self.load_time:.2f}s')
            return True
        except Exception as e:
            print(f'✗ Failed to load {self.model_name}: {e}')
            return False
    
    def generate_recommendation(self, prompt, max_tokens=500):
        if self.model is None:
            return None, 0, 0
        
        start_time = time.time()
        try:
            import torch
            inputs = self.tokenizer.encode(prompt, return_tensors='pt').to(self.device)
            
            outputs = self.model.generate(
                inputs,
                max_new_tokens=max_tokens,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
            
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            inference_time = time.time() - start_time
            token_count = len(outputs[0])
            
            self.inference_times.append(inference_time)
            self.token_counts.append(token_count)
            
            return response, inference_time, token_count
        except Exception as e:
            print(f'Error generating recommendation: {e}')
            return None, 0, 0

print('✅ OpenSourceLLMBackend class defined')

In [ ]:
class GPTBackend(LLMBackend):
    """
    OpenAI GPT backend (requires OPENAI_API_KEY).
    """
    def __init__(self, model_name='gpt-4', api_key=None):
        super().__init__(model_name)
        self.api_key = api_key or os.getenv('OPENAI_API_KEY')
        self.client = None
        self.total_cost = 0
    
    def load_model(self):
        print(f'Initializing {self.model_name} API client...')
        start_time = time.time()
        
        try:
            from openai import OpenAI
            
            if not self.api_key:
                print('⚠ Warning: OPENAI_API_KEY not set')
                return False
            
            self.client = OpenAI(api_key=self.api_key)
            self.load_time = time.time() - start_time
            print(f'✓ {self.model_name} API ready')
            return True
        except Exception as e:
            print(f'✗ Failed to initialize {self.model_name}: {e}')
            return False
    
    def generate_recommendation(self, prompt, max_tokens=500):
        if self.client is None:
            print('Client not initialized')
            return None, 0, 0
        
        start_time = time.time()
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {'role': 'system', 'content': 'You are a book recommendation agent.'},
                    {'role': 'user', 'content': prompt}
                ],
                max_tokens=max_tokens,
                temperature=0.7,
            )
            
            inference_time = time.time() - start_time
            message_content = response.choices[0].message.content
            total_tokens = response.usage.prompt_tokens + response.usage.completion_tokens
            
            self.inference_times.append(inference_time)
            self.token_counts.append(total_tokens)
            
            return message_content, inference_time, total_tokens
        except Exception as e:
            print(f'Error calling OpenAI API: {e}')
            return None, 0, 0

print('✅ GPTBackend class defined')

## Phase 3: Evaluation Metrics

In [ ]:
class RecommendationEvaluator:
    """
    Evaluate LLM recommendations across multiple dimensions.
    """
    def __init__(self, books_df, top_candidates):
        self.books_df = books_df
        self.top_candidates = top_candidates
        self.valid_book_titles = set(books_df['title'].str.lower())
    
    def calculate_hallucination_rate(self, response_text):
        """
        Check if response recommends books outside candidate set.
        """
        candidate_titles = {c.get('title', '').lower() for c in self.top_candidates}
        response_lower = response_text.lower()
        
        hallucinations = 0
        for title in candidate_titles:
            if title in response_lower:
                hallucinations += 1
        
        return 0 if len(candidate_titles) == 0 else (len(candidate_titles) - hallucinations) / len(candidate_titles)
    
    def calculate_context_retention(self, response_text, user_context):
        """
        Score: How well the LLM acknowledged user proficiency and history.
        """
        score = 0
        
        if user_context['primary_proficiency'].lower() in response_text.lower():
            score += 1
        
        recent_titles = [b['title'] for b in user_context['recent_books'][:3]]
        for title in recent_titles:
            if title.lower() in response_text.lower():
                score += 1
                break
        
        return score / 2
    
    def evaluate_response(self, response_text, user_context):
        """
        Comprehensive evaluation of single response.
        """
        if response_text is None:
            return None
        
        metrics = {
            'hallucination_rate': self.calculate_hallucination_rate(response_text),
            'context_retention': self.calculate_context_retention(response_text, user_context),
            'response_length': len(response_text),
            'has_reasoning': 'because' in response_text.lower() or 'reason' in response_text.lower(),
        }
        
        return metrics

print('✅ RecommendationEvaluator class defined')

## Phase 4: Main Evaluation Runner

In [ ]:
class LLMComparativeEvaluation:
    """
    Main evaluation orchestrator for comparing LLM backends.
    """
    def __init__(self, ratings_df, books_df, test_user_ids):
        self.ratings_df = ratings_df
        self.books_df = books_df
        self.test_user_ids = test_user_ids
        self.results = []
        self.llm_backends = {}
    
    def register_backend(self, backend):
        """
        Register an LLM backend for evaluation.
        """
        self.llm_backends[backend.model_name] = backend
    
    def run_evaluation(self, num_candidates=10, max_tokens=500):
        """
        Run complete evaluation across all backends.
        """
        print('\n' + '='*70)
        print('🚀 COMPARATIVE LLM EVALUATION STARTING')
        print('='*70)
        
        print('\n📦 Loading LLM backends...')
        for backend_name, backend in self.llm_backends.items():
            backend.load_model()
        
        for user_id in tqdm(self.test_user_ids, desc='Evaluating users'):
            try:
                user_context = get_user_recommendation_context(user_id, self.ratings_df, self.books_df)
                top_candidates = self.books_df.sample(num_candidates)[['title', 'authors', 'proficiency_level']].to_dict('records')
                prompt = create_llm_prompt_template(user_context, top_candidates)
                evaluator = RecommendationEvaluator(self.books_df, top_candidates)
                
                for backend_name, backend in self.llm_backends.items():
                    response, inference_time, token_count = backend.generate_recommendation(prompt, max_tokens)
                    eval_metrics = evaluator.evaluate_response(response, user_context)
                    
                    if eval_metrics:
                        result = {
                            'user_id': user_id,
                            'llm_model': backend_name,
                            'inference_time_ms': inference_time * 1000,
                            'token_count': token_count,
                            'hallucination_rate': eval_metrics['hallucination_rate'],
                            'context_retention': eval_metrics['context_retention'],
                            'response_length': eval_metrics['response_length'],
                            'has_reasoning': eval_metrics['has_reasoning'],
                        }
                        self.results.append(result)
            except Exception as e:
                print(f'Error evaluating user {user_id}: {e}')
                continue
        
        print('\n✅ Evaluation complete!')
        return pd.DataFrame(self.results)
    
    def generate_summary_report(self, results_df):
        """
        Generate summary statistics by LLM model.
        """
        summary = results_df.groupby('llm_model').agg({
            'inference_time_ms': ['mean', 'std'],
            'token_count': ['mean', 'std'],
            'hallucination_rate': ['mean', 'std'],
            'context_retention': ['mean', 'std'],
            'has_reasoning': 'mean',
        }).round(3)
        
        return summary

print('✅ LLMComparativeEvaluation class defined')

## Phase 5: Visualization Functions

In [ ]:
def plot_comparative_metrics(results_df):
    """
    Create comprehensive comparison visualizations.
    """
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('LLM Comparative Evaluation Results', fontsize=16, fontweight='bold')
    sns.set_style('whitegrid')
    
    if len(results_df) == 0:
        print('No results to visualize')
        return fig
    
    sns.boxplot(data=results_df, x='llm_model', y='inference_time_ms', ax=axes[0, 0])
    axes[0, 0].set_title('Inference Time (ms)')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    sns.boxplot(data=results_df, x='llm_model', y='hallucination_rate', ax=axes[0, 1])
    axes[0, 1].set_title('Hallucination Rate')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    sns.boxplot(data=results_df, x='llm_model', y='context_retention', ax=axes[0, 2])
    axes[0, 2].set_title('Context Retention Score')
    axes[0, 2].tick_params(axis='x', rotation=45)
    
    sns.boxplot(data=results_df, x='llm_model', y='token_count', ax=axes[1, 0])
    axes[1, 0].set_title('Token Usage per Response')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    reasoning_pct = results_df.groupby('llm_model')['has_reasoning'].mean() * 100
    axes[1, 1].bar(reasoning_pct.index, reasoning_pct.values, color='steelblue')
    axes[1, 1].set_title('Responses with Reasoning (%)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].set_ylim([0, 105])
    
    sns.boxplot(data=results_df, x='llm_model', y='response_length', ax=axes[1, 2])
    axes[1, 2].set_title('Response Length (chars)')
    axes[1, 2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    return fig

def create_comparison_matrix(results_df):
    """
    Create detailed comparison matrix.
    """
    summary_stats = results_df.groupby('llm_model').agg({
        'inference_time_ms': 'mean',
        'token_count': 'mean',
        'hallucination_rate': 'mean',
        'context_retention': 'mean',
        'has_reasoning': lambda x: (x.sum() / len(x) * 100),
    }).round(2)
    
    summary_stats.columns = [
        'Avg Inference (ms)',
        'Avg Tokens',
        'Hallucination Rate',
        'Context Score',
        'Reasoning %'
    ]
    
    return summary_stats

print('✅ Visualization functions defined')

## Phase 6: Export Results

In [ ]:
def export_results(results_df, output_dir='./evaluations'):
    """
    Export evaluation results to CSV and JSON.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    csv_path = f'{output_dir}/llm_comparative_evaluation_{timestamp}.csv'
    results_df.to_csv(csv_path, index=False)
    print(f'✓ Detailed results saved to: {csv_path}')
    
    summary_df = create_comparison_matrix(results_df)
    summary_path = f'{output_dir}/llm_comparative_summary_{timestamp}.csv'
    summary_df.to_csv(summary_path)
    print(f'✓ Summary statistics saved to: {summary_path}')
    
    json_path = f'{output_dir}/llm_comparative_evaluation_{timestamp}.json'
    export_data = {
        'timestamp': timestamp,
        'test_users': int(len(results_df['user_id'].unique())),
        'llm_models': results_df['llm_model'].unique().tolist(),
        'summary': summary_df.to_dict(),
    }
    
    with open(json_path, 'w') as f:
        json.dump(export_data, f, indent=2)
    print(f'✓ JSON export saved to: {json_path}')
    
    return csv_path, summary_path, json_path

print('✅ Export functions defined')

## Phase 7: Quick Start Guide

### To run the complete evaluation:

```python
# 1. Create evaluator with small test set
evaluator = LLMComparativeEvaluation(ratings_df, books_df, test_user_ids[:5])

# 2. Register backends to test
evaluator.register_backend(QwenBackend())
# evaluator.register_backend(OpenSourceLLMBackend('Llama-2-7B-Chat', 'meta-llama/Llama-2-7b-chat-hf'))
# evaluator.register_backend(OpenSourceLLMBackend('Mistral-7B', 'mistralai/Mistral-7B-Instruct-v0.2'))
# os.environ['OPENAI_API_KEY'] = 'your-key-here'
# evaluator.register_backend(GPTBackend('gpt-4'))

# 3. Run evaluation
results_df = evaluator.run_evaluation(num_candidates=10, max_tokens=500)

# 4. Generate report
summary = evaluator.generate_summary_report(results_df)
print(summary)

# 5. Visualize
fig = plot_comparative_metrics(results_df)
plt.show()

# 6. Export
export_results(results_df)
```


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║     BookBuddy LLM Comparative Evaluation Framework Ready             ║
║                                                                    ║
║  Framework supports:                                               ║
║  - Qwen2.5-7B-Instruct (baseline)                                  ║
║  - Llama-2-7B, Mistral-7B, Phi-3 (open-source)                    ║
║  - GPT-4, GPT-3.5 (commercial)                                     ║
║                                                                    ║
║  Next steps:                                                        ║
║  1. Register LLM backends in cell below                             ║
║  2. Run evaluation with test users                                  ║
║  3. Generate visualizations and export results                      ║
╚════════════════════════════════════════════════════════════════════╝
""")